# Download image dataset

### Examples
- https://huggingface.co/datasets?task_categories=task_categories:image-classification&sort=trending&search=microsoft
- https://www.tensorflow.org/datasets/catalog/cats_vs_dogs

### Huggingface dataset

In [ ]:
# %pip install datasets

In [3]:
from datasets import load_dataset

In [4]:
# This will download and load the dataset
dataset = load_dataset("microsoft/cats_vs_dogs")

In [5]:
# Check out what's inside
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'labels'],
        num_rows: 23410
    })
})

In [24]:
dataset['train'].features

{'image': Image(mode=None, decode=True, id=None),
 'labels': ClassLabel(names=['cat', 'dog'], id=None)}

In [8]:
dataset['train'].features['labels'].names

['cat', 'dog']

### Tensorflow dataset

In [12]:
# %pip install tensorflow tensorflow-datasets

In [15]:
import tensorflow_datasets as tfds

In [16]:
# Load dataset and split into training (80%), validation (10%), and test (10%)
(train_data, validation_data, test_data), info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True  # This returns the data in (image, label) pairs
)

### This approach will automatically detect unique labels and sample an equal number from each class:

In [21]:
from datasets import load_dataset, DatasetDict, Dataset
import random
from collections import defaultdict

In [22]:
def reduce_dataset(dataset, samples_per_class=500, label_column="labels", seed=42):
    """
    Reduce a dataset to have a balanced number of samples per class.
    
    Args:
        dataset: The Hugging Face dataset (can be a DatasetDict or Dataset)
        samples_per_class: Number of samples to keep per class
        label_column: The column name containing class labels
        seed: Random seed for reproducibility
    
    Returns:
        Reduced dataset with the same structure as the input
    """
    random.seed(seed)
    
    # Handle both Dataset and DatasetDict objects
    if isinstance(dataset, DatasetDict):
        return DatasetDict({
            split: reduce_split(dataset[split], samples_per_class, label_column)
            for split in dataset.keys()
        })
    else:
        return reduce_split(dataset, samples_per_class, label_column)

def reduce_split(split_data, samples_per_class, label_column):
    # Group examples by their labels
    examples_by_class = defaultdict(list)
    
    # Go through all examples and group by label
    for example in split_data:
        label = example[label_column]
        examples_by_class[label].append(example)
    
    reduced_examples = []
    
    # For each class, sample the requested number (or all if fewer are available)
    for label, examples in examples_by_class.items():
        available_samples = len(examples)
        n_samples = min(samples_per_class, available_samples)
        
        sampled_examples = random.sample(examples, n_samples)
        reduced_examples.extend(sampled_examples)
        
        print(f"Class {label}: sampled {n_samples}/{available_samples} examples")
    
    return Dataset.from_list(reduced_examples)

In [23]:
# Example usage
dataset = load_dataset("microsoft/cats_vs_dogs")
reduced_dataset = reduce_dataset(dataset, samples_per_class=500)

Class 0: sampled 500/11741 examples
Class 1: sampled 500/11669 examples
